## Convolution Test Data Generator

In [2]:
import numpy as np
import os

def int_to_hex(val, bit_width):
    if bit_width == 8:
        return f"{val & 0xFF:02x}"
    elif bit_width == 16:
        return f"{val & 0xFFFF:04x}"
    elif bit_width == 32:
        return f"{val & 0xFFFFFFFF:08x}"
    else:
        raise ValueError("Unsupported bit width")

def save_hex_file(data, filepath, bytes_per_line=4, bit_width=8):
    with open(filepath, "w") as f:
        for i in range(0, len(data), bytes_per_line):
            line = data[i:i+bytes_per_line]
            f.write(" ".join([int_to_hex(v, bit_width if data.dtype != np.int32 else 32) for v in line]) + "\n")

def save_decimal_file(data, filepath, bytes_per_line=4):
    with open(filepath, "w") as f:
        for i in range(0, len(data), bytes_per_line):
            line = data[i:i+bytes_per_line]
            f.write(" ".join(map(str, line)) + "\n")

def conv_test_gen(
    ch_in=8,
    ch_out=4,
    input_length=12,
    ker_w=3,
    stride=1,
    pad_left=0,
    pad_right=0,
    use_bias=True,
    data_type=16,
    ifmap_numbers_per_line=2,
    kernel_numbers_per_line=2,
    golden_numbers_per_line=4,
    bias_numbers_per_line=1,
    random_seed=0,
    input_file="ifmap",
    kernel_file="kernel",
    bias_file="bias",
    golden_file="golden",
    output_dir="."  # ⬅️ 新增輸出路徑參數
):
    os.makedirs(output_dir, exist_ok=True)
    np.random.seed(random_seed)

    # === 資料型態設定 ===
    if data_type == 8:
        dtype = np.int8
        low, high = -128, 127
    elif data_type == 16:
        dtype = np.int16
        low, high = -256, 256 ## -32768, 32767 is too large
    else:
        raise ValueError("Only 8 or 16-bit supported.")

    # === 資料生成 ===
    input_data = np.random.randint(low, high, size=(ch_in, input_length)).astype(dtype)
    kernel = np.random.randint(low, high, size=(ch_out, ch_in, ker_w)).astype(dtype)
    bias = np.random.randint(-32768, 32767, size=(ch_out,), dtype=np.int32) if use_bias else np.zeros(ch_out, dtype=np.int32)

    # === 輸出 input ===
    input_interleaved = input_data.T.flatten()
    save_hex_file(input_interleaved, os.path.join(output_dir, f"{input_file}_hex.txt"), bytes_per_line=ifmap_numbers_per_line, bit_width=data_type)
    save_decimal_file(input_interleaved, os.path.join(output_dir, f"{input_file}_dec.txt"), bytes_per_line=ifmap_numbers_per_line)

    # === 輸出 kernel ===
    kernel_flat = kernel.transpose(0, 2, 1).reshape(-1)
    save_hex_file(kernel_flat, os.path.join(output_dir, f"{kernel_file}_hex.txt"), bytes_per_line=kernel_numbers_per_line, bit_width=data_type)
    save_decimal_file(kernel_flat, os.path.join(output_dir, f"{kernel_file}_dec.txt"), bytes_per_line=kernel_numbers_per_line)

    # === 輸出 bias（如有）===
    if use_bias:
        save_hex_file(bias, os.path.join(output_dir, f"{bias_file}_hex.txt"), bytes_per_line=bias_numbers_per_line, bit_width=32)
        save_decimal_file(bias, os.path.join(output_dir, f"{bias_file}_dec.txt"), bytes_per_line=bias_numbers_per_line)

    # === Padding ===
    padded_input = np.pad(input_data, ((0, 0), (pad_left, pad_right)), mode='constant', constant_values=0)
    L_in = padded_input.shape[1]
    out_len = (L_in - ker_w) // stride + 1
    output = np.zeros((ch_out, out_len), dtype=np.int32)

    # === Convolution ===
    for oc in range(ch_out):
        for i in range(out_len):
            acc = 0
            for ic in range(ch_in):
                window = padded_input[ic, i*stride:i*stride+ker_w]
                acc += np.sum(window.astype(np.int32) * kernel[oc, ic].astype(np.int32))
            acc += bias[oc]
            output[oc, i] = acc

    output_flat = output.T.flatten()
    save_hex_file(output_flat, os.path.join(output_dir, f"{golden_file}_hex.txt"), bytes_per_line=golden_numbers_per_line, bit_width=32)
    save_decimal_file(output_flat, os.path.join(output_dir, f"{golden_file}_dec.txt"), bytes_per_line=golden_numbers_per_line)

    print("✅ 檔案產生完成")
    print(f"📁 輸出路徑: {os.path.abspath(output_dir)}")


## PE INFO Generator

In [6]:
import os

def parse_value(v):
    return int(v, 0) if isinstance(v, str) else v

def instruction_gen(
    total_bits=32,
    fields=[
        ("opcode", 4, "0xB"),
        ("imm", 8, 123),
        ("flag", 1, 1),
        ("manual", 24, 31, "0xAB")
    ],
    output_file="custom_hex.txt",
    output_dir="."  # ⬅️ 新增：可自訂輸出資料夾
):
    os.makedirs(output_dir, exist_ok=True)
    full_path = os.path.join(output_dir, output_file)

    value = 0
    curr_bit = 0

    for entry in fields:
        if len(entry) == 3:
            name, width, val = entry
            start_bit = curr_bit
            end_bit = curr_bit + width - 1
            curr_bit += width
        elif len(entry) == 4:
            name, start_bit, end_bit, val = entry
            width = end_bit - start_bit + 1
        else:
            raise ValueError(f"欄位格式錯誤: {entry}")

        val = parse_value(val)
        if val >= (1 << width):
            raise ValueError(f"欄位 {name} 值 {val} 超出 {width} bits 上限")

        value |= (val & ((1 << width) - 1)) << start_bit

    hex_str = f"{value:0{(total_bits + 3) // 4}x}"
    with open(full_path, "w") as f:
        f.write(hex_str + "\n")

    print(f"✅ 已產生 {full_path}")


## ECG v2

### ECG v2 conv 1

In [48]:
## bits number
_DLA_MODE       = 2
_DLA_FMP        = 1
_PE_ID          = 4
_DLA_PAD_W      = 2
_DLA_MP_W       = 4
_DLA_CH         = 5
_IFM_MODE       = 2
_DLA_KER_A      = 13
_DLA_IFM_A      = 12
_DLA_IFM_L      = 10
## parameter
dla_mode        = 0
dla_relu        = 1
dla_fmp         = 0
dla_pe_active   = 15 ## -1
dla_pad_l       = 0
# # dla_pad_r       = 1
dla_mp_w        = 0
dla_ap          = 0
dla_d_type      = 1 ## 0: 8bits; 1: 16 bits

ifm_mode        = 1
dla_ch_out      = 15 ## -1
ifm_group_num   = 0
ifm_group_ch_in = 0 ## ch in num of one ifm group
ker_w_part      = 2 ## 3-1
ker_load_times  = 0 ## 1-0
dla_scenario    = 0

dla_ker_a_start = 0
dla_ifm_a_start = 0
dla_ifm_a_end   = 510 ## 127*4 + 2 ## because this layer address + 2
dla_ofm_a_start = 0
dla_ofm_a_jump  = 1 # 4
dla_ofm_a_end   = 1023 # 4092 ## 1023*4 

bias_is_used    = 1 ## 0: no bias; 1: use bias
scale_is_used   = 0 ## 0: no scale; 1: use scale


DIR_O           = "./ecg_v2_conv1"
fields=[
    ("dla_mode", _DLA_MODE, str(int(dla_mode))),
    ("dla_relu", 1, str(int(dla_relu))),
    ("dla_fmp", _DLA_FMP, str(int(dla_fmp))),
    ("dla_pe_active", _PE_ID, str(int(dla_pe_active))),
    ("dla_pad_l", _DLA_PAD_W, str(int(dla_pad_l))),
    # ("dla_pad_r", _DLA_PAD_W, str(int(dla_pad_r))),
    ("dla_mp_w", _DLA_MP_W, str(int(dla_mp_w))),
    ("dla_ap", 1, str(int(dla_ap))),
    ("dla_d_type", 1, str(int(dla_d_type))),
    ("ifm_mode", _IFM_MODE, str(int(ifm_mode))),
    ("dla_ch_out", _DLA_CH, str(int(dla_ch_out))),
    ("ifm_group_num", _PE_ID, str(int(ifm_group_num))),
    ("ifm_group_ch_in", _DLA_CH, str(int(ifm_group_ch_in))),
    ("ker_w_part", _DLA_CH, str(int(ker_w_part))),
    ("ker_load_times", _DLA_CH, str(int(ker_load_times))),
    ("dla_scenario", _DLA_MODE, str(dla_scenario)),
    ("dla_ker_a_start", _DLA_KER_A, str(int(dla_ker_a_start))),
    ("dla_ifm_a_start", _DLA_IFM_A, str(int(dla_ifm_a_start))),
    ("dla_ifm_a_end", _DLA_IFM_A, str(int(dla_ifm_a_end))),
    ("dla_ofm_a_start", _DLA_IFM_L, str(int(dla_ofm_a_start))),
    ("dla_ofm_a_jump", _DLA_IFM_L, str(int(dla_ofm_a_jump))),
    ("dla_ofm_a_end", _DLA_IFM_L, str(int(dla_ofm_a_end))),
    ("bias_is_used", 1, str(bias_is_used)),
    ("scale_is_used", 1, str(scale_is_used))
]
instruction_gen(
    total_bits=128,
    fields=fields,
    output_file="dla_info.txt",
    output_dir=DIR_O
)
real_bits = sum(width for _, width, _ in fields)
print("real_bits: ", real_bits)

✅ 已產生 ./ecg_v2_conv1\dla_info.txt
real_bits:  113


In [24]:
_ifm_id         = 4
_stride         = 4
_int16_is       = 1
_pe_out_en      = 1
_pe_mode        = 1
_ker_w          = 6

ker_w           = 2 # 3-1
stride          = 2
int16_is        = 1
pe_mode         = 0
#### PE INFO
DIR_O = "./ecg_v2_conv1"
instruction_gen(
    total_bits =
        _ifm_id + _ker_w + _stride + _int16_is +
        _pe_out_en, ## 32
    fields=[
        ("ifm_id", _ifm_id, "0"),
        ("stride", _stride, str(int(stride))),
        ("int16_is", _int16_is, str(int(int16_is))),
        ("pe_out_en", _pe_out_en, "0"),
        ("pe_mode", _pe_mode, str(int(pe_mode))),
        ("ker_w", _ker_w, str(int(ker_w))),
    ],
    output_file="pe0.txt",
    output_dir=DIR_O
)
instruction_gen(
    total_bits =
        _ifm_id + _ker_w + _stride + _int16_is +
        _pe_out_en, ## 32
    fields=[
        ("ifm_id", _ifm_id, "0"),
        ("stride", _stride, str(int(stride))),
        ("int16_is", _int16_is, str(int(int16_is))),
        ("pe_out_en", _pe_out_en, "0"),
        ("pe_mode", _pe_mode, str(int(pe_mode))),
        ("ker_w", _ker_w, str(int(ker_w))),
    ],
    output_file="pe1.txt",
    output_dir=DIR_O
)

✅ 已產生 ./ecg_v2_conv1\pe0.txt
✅ 已產生 ./ecg_v2_conv1\pe1.txt


### ECG v2 conv 2 p0

In [49]:
DIR_O = "./ecg_v2_conv2_p0"
## bits number
_DLA_MODE       = 2
_DLA_FMP        = 1
_PE_ID          = 4
_DLA_PAD_W      = 2
_DLA_MP_W       = 4
_DLA_CH         = 5
_IFM_MODE       = 2
_DLA_KER_A      = 13
_DLA_IFM_A      = 12
_DLA_IFM_L      = 10
## parameter
dla_mode        = 0
dla_relu        = 1
dla_fmp         = 0
dla_pe_active   = 15
dla_pad_l       = 0
# # dla_pad_r       = 1
dla_mp_w        = 0
dla_ap          = 0
dla_d_type      = 1 # 0: 8bits; 1: 16 bits

dla_ch_out      = 3
ifm_mode        = 2
ifm_group_num   = 3
ifm_group_ch_in = 1 # ch in num of one ifm group
ker_w_part      = 1
ker_load_times  = 2 # 3-1
dla_scenario    = 2 # 0: 1*16; 1: 2*8; 2: 4*4; 3: 8*2

dla_ker_a_start = 160
dla_ifm_a_start = 0
dla_ifm_a_end   = 4092 # 1023*4
dla_ofm_a_start = 0
dla_ofm_a_jump  = 7 # ((16-4)/2 + 1) * 4
dla_ofm_a_end   = 505# 2020 # 4072 # (0 + 4/2 + 16/2*(128-1)) *4

bias_is_used    = 1 # 0: no bias; 1: use bias
scale_is_used   = 0 # 0: no scale; 1: use scale

fields=[
    ("dla_mode", _DLA_MODE, str(int(dla_mode))),
    ("dla_relu", 1, str(int(dla_relu))),
    ("dla_fmp", _DLA_FMP, str(int(dla_fmp))),
    ("dla_pe_active", _PE_ID, str(int(dla_pe_active))),
    ("dla_pad_l", _DLA_PAD_W, str(int(dla_pad_l))),
    # ("dla_pad_r", _DLA_PAD_W, str(int(dla_pad_r))),
    ("dla_mp_w", _DLA_MP_W, str(int(dla_mp_w))),
    ("dla_ap", 1, str(int(dla_ap))),
    ("dla_d_type", 1, str(int(dla_d_type))),
    ("ifm_mode", _IFM_MODE, str(int(ifm_mode))),
    ("dla_ch_out", _DLA_CH, str(int(dla_ch_out))),
    ("ifm_group_num", _PE_ID, str(int(ifm_group_num))),
    ("ifm_group_ch_in", _DLA_CH, str(int(ifm_group_ch_in))),
    ("ker_w_part", _DLA_CH, str(int(ker_w_part))),
    ("ker_load_times", _DLA_CH, str(int(ker_load_times))),
    ("dla_scenario", _DLA_MODE, str(dla_scenario)),
    ("dla_ker_a_start", _DLA_KER_A, str(int(dla_ker_a_start))),
    ("dla_ifm_a_start", _DLA_IFM_A, str(int(dla_ifm_a_start))),
    ("dla_ifm_a_end", _DLA_IFM_A, str(int(dla_ifm_a_end))),
    ("dla_ofm_a_start", _DLA_IFM_L, str(int(dla_ofm_a_start))),
    ("dla_ofm_a_jump", _DLA_IFM_L, str(int(dla_ofm_a_jump))),
    ("dla_ofm_a_end", _DLA_IFM_L, str(int(dla_ofm_a_end))),
    ("bias_is_used", 1, str(bias_is_used)),  ## 0: no bias; 1: use bias
    ("scale_is_used", 1, str(scale_is_used))  ## 0: no scale; 1: use scale
]
instruction_gen(
    total_bits=128,
    fields=fields,
    output_file="dla_info.txt",
    output_dir=DIR_O
)
real_bits = sum(width for _, width, _ in fields)
print("real_bits: ", real_bits)

✅ 已產生 ./ecg_v2_conv2_p0\dla_info.txt
real_bits:  113


In [26]:
DIR_O = "./ecg_v2_conv2_p0"
_ifm_id         = 4
_stride         = 4
_int16_is       = 1
_pe_out_en      = 1
_pe_mode        = 1
_ker_w          = 6

ker_w           = 5 # 3*2-1
stride          = 4
int16_is        = 1
pe_mode         = 0
#### PE INFO
instruction_gen(
    total_bits = 32, ## _ifm_id + _ker_w + _stride + _int16_is +_pe_out_en
    fields=[
        ("ifm_id", _ifm_id, "0"),
        ("stride", _stride, str(int(stride))),
        ("int16_is", _int16_is, str(int(int16_is))),
        ("pe_out_en", _pe_out_en, "1"),
        ("pe_mode", _pe_mode, str(int(pe_mode))),
        ("ker_w", _ker_w, str(int(ker_w)))
    ],
    output_file="pe0.txt",
    output_dir=DIR_O
)
instruction_gen(
    total_bits = 32, ## _ifm_id + _ker_w + _stride + _int16_is +_pe_out_en
    fields=[
        ("ifm_id", _ifm_id, "1"),
        ("stride", _stride, str(int(stride))),
        ("int16_is", _int16_is, str(int(int16_is))),
        ("pe_out_en", _pe_out_en, "1"),
        ("pe_mode", _pe_mode, str(int(pe_mode))),
        ("ker_w", _ker_w, str(int(ker_w)))
    ],
    output_file="pe1.txt",
    output_dir=DIR_O
)
instruction_gen(
    total_bits = 32, ## _ifm_id + _ker_w + _stride + _int16_is +_pe_out_en
    fields=[
        ("ifm_id", _ifm_id, "3"),
        ("stride", _stride, str(int(stride))),
        ("int16_is", _int16_is, str(int(int16_is))),
        ("pe_out_en", _pe_out_en, "0"),
        ("pe_mode", _pe_mode, str(int(pe_mode))),
        ("ker_w", _ker_w, str(int(ker_w)))
    ],
    output_file="pe3.txt",
    output_dir=DIR_O
)

✅ 已產生 ./ecg_v2_conv2_p0\pe0.txt
✅ 已產生 ./ecg_v2_conv2_p0\pe1.txt
✅ 已產生 ./ecg_v2_conv2_p0\pe3.txt


### ECG v2 conv2 p1~p3

In [50]:
DIR_O = "./ecg_v2_conv2_p1"
## bits number
_DLA_MODE       = 2
_DLA_FMP        = 1
_PE_ID          = 4
_DLA_PAD_W      = 2
_DLA_MP_W       = 4
_DLA_CH         = 5
_IFM_MODE       = 2
_DLA_KER_A      = 13
_DLA_IFM_A      = 12
_DLA_IFM_L      = 10
## parameter
dla_mode        = 0
dla_relu        = 1
dla_fmp         = 0
dla_pe_active   = 15
dla_pad_l       = 0
# # dla_pad_r       = 1
dla_mp_w        = 0
dla_ap          = 0
dla_d_type      = 1 # 0: 8bits; 1: 16 bits

dla_ch_out      = 3
ifm_mode        = 2
ifm_group_num   = 3
ifm_group_ch_in = 1 # ch in num of one ifm group
ker_w_part      = 1
ker_load_times  = 2 # 3-1
dla_scenario    = 2 # 0: 1*16; 1: 2*8; 2: 4*4; 3: 8*2

dla_ker_a_start = 560
dla_ifm_a_start = 0
dla_ifm_a_end   = 4092 # 1023*4
dla_ofm_a_start = 0
dla_ofm_a_jump  = 7 # 28 # ((16-4)/2 + 1) * 4
dla_ofm_a_end   = 505 # 2020 # 4072 # (0 + 4/2 + 16/2*(128-1)) *4

bias_is_used    = 1 # 0: no bias; 1: use bias
scale_is_used   = 0 # 0: no scale; 1: use scale

fields=[
    ("dla_mode", _DLA_MODE, str(int(dla_mode))),
    ("dla_relu", 1, str(int(dla_relu))),
    ("dla_fmp", _DLA_FMP, str(int(dla_fmp))),
    ("dla_pe_active", _PE_ID, str(int(dla_pe_active))),
    ("dla_pad_l", _DLA_PAD_W, str(int(dla_pad_l))),
    # ("dla_pad_r", _DLA_PAD_W, str(int(dla_pad_r))),
    ("dla_mp_w", _DLA_MP_W, str(int(dla_mp_w))),
    ("dla_ap", 1, str(int(dla_ap))),
    ("dla_d_type", 1, str(int(dla_d_type))),
    ("ifm_mode", _IFM_MODE, str(int(ifm_mode))),
    ("dla_ch_out", _DLA_CH, str(int(dla_ch_out))),
    ("ifm_group_num", _PE_ID, str(int(ifm_group_num))),
    ("ifm_group_ch_in", _DLA_CH, str(int(ifm_group_ch_in))),
    ("ker_w_part", _DLA_CH, str(int(ker_w_part))),
    ("ker_load_times", _DLA_CH, str(int(ker_load_times))),
    ("dla_scenario", _DLA_MODE, str(dla_scenario)),
    ("dla_ker_a_start", _DLA_KER_A, str(int(dla_ker_a_start))),
    ("dla_ifm_a_start", _DLA_IFM_A, str(int(dla_ifm_a_start))),
    ("dla_ifm_a_end", _DLA_IFM_A, str(int(dla_ifm_a_end))),
    ("dla_ofm_a_start", _DLA_IFM_L, str(int(dla_ofm_a_start))),
    ("dla_ofm_a_jump", _DLA_IFM_L, str(int(dla_ofm_a_jump))),
    ("dla_ofm_a_end", _DLA_IFM_L, str(int(dla_ofm_a_end))),
    ("bias_is_used", 1, str(bias_is_used)),  ## 0: no bias; 1: use bias
    ("scale_is_used", 1, str(scale_is_used))  ## 0: no scale; 1: use scale
]
instruction_gen(
    total_bits=128,
    fields=fields,
    output_file="dla_info.txt",
    output_dir=DIR_O
)
real_bits = sum(width for _, width, _ in fields)
print("real_bits: ", real_bits)

✅ 已產生 ./ecg_v2_conv2_p1\dla_info.txt
real_bits:  113


In [52]:
DIR_O = "./ecg_v2_conv2_p2"
## bits number
_DLA_MODE       = 2
_DLA_FMP        = 1
_PE_ID          = 4
_DLA_PAD_W      = 2
_DLA_MP_W       = 4
_DLA_CH         = 5
_IFM_MODE       = 2
_DLA_KER_A      = 13
_DLA_IFM_A      = 12
_DLA_IFM_L      = 10
## parameter
dla_mode        = 0
dla_relu        = 1
dla_fmp         = 0
dla_pe_active   = 15
dla_pad_l       = 0
# # dla_pad_r       = 1
dla_mp_w        = 0
dla_ap          = 0
dla_d_type      = 1 # 0: 8bits; 1: 16 bits

dla_ch_out      = 3
ifm_mode        = 2
ifm_group_num   = 3
ifm_group_ch_in = 1 # ch in num of one ifm group
ker_w_part      = 1
ker_load_times  = 2 # 3-1
dla_scenario    = 2 # 0: 1*16; 1: 2*8; 2: 4*4; 3: 8*2

dla_ker_a_start = 960
dla_ifm_a_start = 0
dla_ifm_a_end   = 4092 # 1023*4
dla_ofm_a_start = 0
dla_ofm_a_jump  = 7 # ((16-4)/2 + 1) * 4
dla_ofm_a_end   = 505# 2020 # 4072 # (0 + 4/2 + 16/2*(128-1)) *4

bias_is_used    = 1 # 0: no bias; 1: use bias
scale_is_used   = 0 # 0: no scale; 1: use scale

fields=[
    ("dla_mode", _DLA_MODE, str(int(dla_mode))),
    ("dla_relu", 1, str(int(dla_relu))),
    ("dla_fmp", _DLA_FMP, str(int(dla_fmp))),
    ("dla_pe_active", _PE_ID, str(int(dla_pe_active))),
    ("dla_pad_l", _DLA_PAD_W, str(int(dla_pad_l))),
    # ("dla_pad_r", _DLA_PAD_W, str(int(dla_pad_r))),
    ("dla_mp_w", _DLA_MP_W, str(int(dla_mp_w))),
    ("dla_ap", 1, str(int(dla_ap))),
    ("dla_d_type", 1, str(int(dla_d_type))),
    ("ifm_mode", _IFM_MODE, str(int(ifm_mode))),
    ("dla_ch_out", _DLA_CH, str(int(dla_ch_out))),
    ("ifm_group_num", _PE_ID, str(int(ifm_group_num))),
    ("ifm_group_ch_in", _DLA_CH, str(int(ifm_group_ch_in))),
    ("ker_w_part", _DLA_CH, str(int(ker_w_part))),
    ("ker_load_times", _DLA_CH, str(int(ker_load_times))),
    ("dla_scenario", _DLA_MODE, str(dla_scenario)),
    ("dla_ker_a_start", _DLA_KER_A, str(int(dla_ker_a_start))),
    ("dla_ifm_a_start", _DLA_IFM_A, str(int(dla_ifm_a_start))),
    ("dla_ifm_a_end", _DLA_IFM_A, str(int(dla_ifm_a_end))),
    ("dla_ofm_a_start", _DLA_IFM_L, str(int(dla_ofm_a_start))),
    ("dla_ofm_a_jump", _DLA_IFM_L, str(int(dla_ofm_a_jump))),
    ("dla_ofm_a_end", _DLA_IFM_L, str(int(dla_ofm_a_end))),
    ("bias_is_used", 1, str(bias_is_used)),  ## 0: no bias; 1: use bias
    ("scale_is_used", 1, str(scale_is_used))  ## 0: no scale; 1: use scale
]
instruction_gen(
    total_bits=128,
    fields=fields,
    output_file="dla_info.txt",
    output_dir=DIR_O
)
real_bits = sum(width for _, width, _ in fields)
print("real_bits: ", real_bits)

✅ 已產生 ./ecg_v2_conv2_p2\dla_info.txt
real_bits:  113


In [51]:
DIR_O = "./ecg_v2_conv2_p3"
## bits number
_DLA_MODE       = 2
_DLA_FMP        = 1
_PE_ID          = 4
_DLA_PAD_W      = 2
_DLA_MP_W       = 4
_DLA_CH         = 5
_IFM_MODE       = 2
_DLA_KER_A      = 13
_DLA_IFM_A      = 12
_DLA_IFM_L      = 10
## parameter
dla_mode        = 0
dla_relu        = 1
dla_fmp         = 0
dla_pe_active   = 15
dla_pad_l       = 0
# # dla_pad_r       = 1
dla_mp_w        = 0
dla_ap          = 0
dla_d_type      = 1 # 0: 8bits; 1: 16 bits

dla_ch_out      = 3
ifm_mode        = 2
ifm_group_num   = 3
ifm_group_ch_in = 1 # ch in num of one ifm group
ker_w_part      = 1
ker_load_times  = 2 # 3-1
dla_scenario    = 2 # 0: 1*16; 1: 2*8; 2: 4*4; 3: 8*2

dla_ker_a_start = 1360
dla_ifm_a_start = 0
dla_ifm_a_end   = 4092 # 1023*4
dla_ofm_a_start = 0
dla_ofm_a_jump  = 7 # ((16-4)/2 + 1) * 4
dla_ofm_a_end   = 505 # 2020 # 4072 # (0 + 4/2 + 16/2*(128-1)) *4

bias_is_used    = 1 # 0: no bias; 1: use bias
scale_is_used   = 0 # 0: no scale; 1: use scale

fields=[
    ("dla_mode", _DLA_MODE, str(int(dla_mode))),
    ("dla_relu", 1, str(int(dla_relu))),
    ("dla_fmp", _DLA_FMP, str(int(dla_fmp))),
    ("dla_pe_active", _PE_ID, str(int(dla_pe_active))),
    ("dla_pad_l", _DLA_PAD_W, str(int(dla_pad_l))),
    # ("dla_pad_r", _DLA_PAD_W, str(int(dla_pad_r))),
    ("dla_mp_w", _DLA_MP_W, str(int(dla_mp_w))),
    ("dla_ap", 1, str(int(dla_ap))),
    ("dla_d_type", 1, str(int(dla_d_type))),
    ("ifm_mode", _IFM_MODE, str(int(ifm_mode))),
    ("dla_ch_out", _DLA_CH, str(int(dla_ch_out))),
    ("ifm_group_num", _PE_ID, str(int(ifm_group_num))),
    ("ifm_group_ch_in", _DLA_CH, str(int(ifm_group_ch_in))),
    ("ker_w_part", _DLA_CH, str(int(ker_w_part))),
    ("ker_load_times", _DLA_CH, str(int(ker_load_times))),
    ("dla_scenario", _DLA_MODE, str(dla_scenario)),
    ("dla_ker_a_start", _DLA_KER_A, str(int(dla_ker_a_start))),
    ("dla_ifm_a_start", _DLA_IFM_A, str(int(dla_ifm_a_start))),
    ("dla_ifm_a_end", _DLA_IFM_A, str(int(dla_ifm_a_end))),
    ("dla_ofm_a_start", _DLA_IFM_L, str(int(dla_ofm_a_start))),
    ("dla_ofm_a_jump", _DLA_IFM_L, str(int(dla_ofm_a_jump))),
    ("dla_ofm_a_end", _DLA_IFM_L, str(int(dla_ofm_a_end))),
    ("bias_is_used", 1, str(bias_is_used)),  ## 0: no bias; 1: use bias
    ("scale_is_used", 1, str(scale_is_used))  ## 0: no scale; 1: use scale
]
instruction_gen(
    total_bits=128,
    fields=fields,
    output_file="dla_info.txt",
    output_dir=DIR_O
)
real_bits = sum(width for _, width, _ in fields)
print("real_bits: ", real_bits)

✅ 已產生 ./ecg_v2_conv2_p3\dla_info.txt
real_bits:  113


### ECG v2 fc1

In [ ]:
DIR_O = "./ecg_v2_fc1"
## bits number
_DLA_MODE       = 2
_DLA_FMP        = 1
_PE_ID          = 4
_DLA_PAD_W      = 2
_DLA_MP_W       = 4
_DLA_CH         = 5
_IFM_MODE       = 2
_DLA_KER_A      = 13
_DLA_IFM_A      = 12
_DLA_IFM_L      = 10
## parameter
dla_mode        = 1 # fully connect
dla_relu        = 0
dla_fmp         = 0
dla_pe_active   = 9 # 10-1
dla_pad_l       = 0
# dla_pad_r       = 0
dla_mp_w        = 0
dla_ap          = 0
dla_d_type      = 1 # 0: 8bits; 1: 16 bits

dla_ch_out      = 9 # 10-1
ifm_mode        = 2
ifm_group_num   = 0
ifm_group_ch_in = 4 # ch in num of one ifm group # 5-1
ker_w_part      = 0 # 1-1
ker_load_times  = 0 # 1-1 ## 不影響功能
dla_scenario    = 0 # 0: 1*16; 1: 2*8; 2: 4*4; 3: 8*2

dla_ker_a_start = 0
dla_ifm_a_start = 0
dla_ifm_a_end   = 140 # (36-1)*4
dla_ofm_a_start = 0
dla_ofm_a_jump  = 1 # 1 * 4
dla_ofm_a_end   = 16 # (5-1)*4

bias_is_used    = 1 # 0: no bias; 1: use bias
scale_is_used   = 0 # 0: no scale; 1: use scale

fields=[
    ("dla_mode", _DLA_MODE, str(int(dla_mode))),
    ("dla_relu", 1, str(int(dla_relu))),
    ("dla_fmp", _DLA_FMP, str(int(dla_fmp))),
    ("dla_pe_active", _PE_ID, str(int(dla_pe_active))),
    ("dla_pad_l", _DLA_PAD_W, str(int(dla_pad_l))),
    # ("dla_pad_r", _DLA_PAD_W, str(int(dla_pad_r))),
    ("dla_mp_w", _DLA_MP_W, str(int(dla_mp_w))),
    ("dla_ap", 1, str(int(dla_ap))),
    ("dla_d_type", 1, str(int(dla_d_type))),
    ("ifm_mode", _IFM_MODE, str(int(ifm_mode))),
    ("dla_ch_out", _DLA_CH, str(int(dla_ch_out))),
    ("ifm_group_num", _PE_ID, str(int(ifm_group_num))),
    ("ifm_group_ch_in", _DLA_CH, str(int(ifm_group_ch_in))),
    ("ker_w_part", _DLA_CH, str(int(ker_w_part))),
    ("ker_load_times", _DLA_CH, str(int(ker_load_times))),
    ("dla_scenario", _DLA_MODE, str(dla_scenario)),
    ("dla_ker_a_start", _DLA_KER_A, str(int(dla_ker_a_start))),
    ("dla_ifm_a_start", _DLA_IFM_A, str(int(dla_ifm_a_start))),
    ("dla_ifm_a_end", _DLA_IFM_A, str(int(dla_ifm_a_end))),
    ("dla_ofm_a_start", _DLA_IFM_L, str(int(dla_ofm_a_start))),
    ("dla_ofm_a_jump", _DLA_IFM_L, str(int(dla_ofm_a_jump))),
    ("dla_ofm_a_end", _DLA_IFM_L, str(int(dla_ofm_a_end))),
    ("bias_is_used", 1, str(bias_is_used)),  ## 0: no bias; 1: use bias
    ("scale_is_used", 1, str(scale_is_used))  ## 0: no scale; 1: use scale
]
instruction_gen(
    total_bits=128,
    fields=fields,
    output_file="dla_info.txt",
    output_dir=DIR_O
)
real_bits = sum(width for _, width, _ in fields)
print("real_bits: ", real_bits)

✅ 已產生 ./ecg_v2_fc1\dla_info.txt
real_bits:  119


In [34]:
DIR_O = "./ecg_v2_fc1"
_ifm_id         = 4
_stride         = 4
_int16_is       = 1
_pe_out_en      = 1
_ker_w          = 6
_pe_mode        = 1

ker_w           = 35 # 1*36-1
stride          = 0
int16_is        = 1
pe_mode         = 1
#### PE INFO
instruction_gen(
    total_bits = 32, ## _ifm_id + _ker_w + _stride + _int16_is +_pe_out_en
    fields=[
        ("ifm_id", _ifm_id, "0"),
        ("stride", _stride, str(int(stride))),
        ("int16_is", _int16_is, str(int(int16_is))),
        ("pe_out_en", _pe_out_en, "0"),
        ("pe_mode", _pe_mode, str(int(pe_mode))),
        ("ker_w", _ker_w, str(int(ker_w)))
    ],
    output_file="pe0.txt",
    output_dir=DIR_O
)

✅ 已產生 ./ecg_v2_fc1\pe0.txt


### ECG v2 fc2

In [ ]:
DIR_O = "./ecg_v2_fc2"
## bits number
_DLA_MODE       = 2
_DLA_FMP        = 1
_PE_ID          = 4
_DLA_PAD_W      = 2
_DLA_MP_W       = 4
_DLA_CH         = 5
_IFM_MODE       = 2
_DLA_KER_A      = 13
_DLA_IFM_A      = 12
_DLA_IFM_L      = 10
## parameter
dla_mode        = 1
dla_relu        = 0
dla_fmp         = 0
dla_pe_active   = 5 # 6-1
dla_pad_l       = 0
# dla_pad_r       = 0
dla_mp_w        = 0
dla_ap          = 0
dla_d_type      = 1 # 0: 8bits; 1: 16 bits

dla_ch_out      = 5 # 6-1
ifm_mode        = 2
ifm_group_num   = 0
ifm_group_ch_in = 4 # ch in num of one ifm group # 5-1
ker_w_part      = 4 # 5-1
ker_load_times  = 0 # 1-1
dla_scenario    = 0 # 0: 1*16; 1: 2*8; 2: 4*4; 3: 8*2

dla_ker_a_start = 0
dla_ifm_a_start = 0
dla_ifm_a_end   = 16 # 4*4
dla_ofm_a_start = 0
dla_ofm_a_jump  = 1 # 1 * 4
dla_ofm_a_end   = 8 # 2*4

bias_is_used    = 1 # 0: no bias; 1: use bias
scale_is_used   = 0 # 0: no scale; 1: use scale

fields=[
    ("dla_mode", _DLA_MODE, str(int(dla_mode))),
    ("dla_relu", 1, str(int(dla_relu))),
    ("dla_fmp", _DLA_FMP, str(int(dla_fmp))),
    ("dla_pe_active", _PE_ID, str(int(dla_pe_active))),
    ("dla_pad_l", _DLA_PAD_W, str(int(dla_pad_l))),
    # ("dla_pad_r", _DLA_PAD_W, str(int(dla_pad_r))),
    ("dla_mp_w", _DLA_MP_W, str(int(dla_mp_w))),
    ("dla_ap", 1, str(int(dla_ap))),
    ("dla_d_type", 1, str(int(dla_d_type))),
    ("ifm_mode", _IFM_MODE, str(int(ifm_mode))),
    ("dla_ch_out", _DLA_CH, str(int(dla_ch_out))),
    ("ifm_group_num", _PE_ID, str(int(ifm_group_num))),
    ("ifm_group_ch_in", _DLA_CH, str(int(ifm_group_ch_in))),
    ("ker_w_part", _DLA_CH, str(int(ker_w_part))),
    ("ker_load_times", _DLA_CH, str(int(ker_load_times))),
    ("dla_scenario", _DLA_MODE, str(dla_scenario)),
    ("dla_ker_a_start", _DLA_KER_A, str(int(dla_ker_a_start))),
    ("dla_ifm_a_start", _DLA_IFM_A, str(int(dla_ifm_a_start))),
    ("dla_ifm_a_end", _DLA_IFM_A, str(int(dla_ifm_a_end))),
    ("dla_ofm_a_start", _DLA_IFM_L, str(int(dla_ofm_a_start))),
    ("dla_ofm_a_jump", _DLA_IFM_L, str(int(dla_ofm_a_jump))),
    ("dla_ofm_a_end", _DLA_IFM_L, str(int(dla_ofm_a_end))),
    ("bias_is_used", 1, str(bias_is_used)),  ## 0: no bias; 1: use bias
    ("scale_is_used", 1, str(scale_is_used))  ## 0: no scale; 1: use scale
]
instruction_gen(
    total_bits=128,
    fields=fields,
    output_file="dla_info.txt",
    output_dir=DIR_O
)
real_bits = sum(width for _, width, _ in fields)
print("real_bits: ", real_bits)

✅ 已產生 ./ecg_v2_fc2\dla_info.txt
real_bits:  119


In [ ]:
def convert_hex_by_scan(input_path, output_path, scan_lines=4):
    # 讀取輸入
    with open(input_path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]

    # 每行拆成 4 字元一組
    split_lines = [[line[i:i+4] for i in range(0, len(line), 4)] for line in lines]

    result = []

    # 每次處理 scan_lines 行
    for i in range(0, len(split_lines), scan_lines):
        group = split_lines[i:i+scan_lines]
        if len(group) < scan_lines:
            break  # 不足一組就跳出

        num_cols = len(group[0])
        # 對每一欄操作
        for col in range(num_cols):
            # 每兩行一組（下 + 上）
            for row in range(0, scan_lines, 2):
                high = group[row+1][col]
                low = group[row][col]
                result.append(high + low)

    # 輸出
    with open(output_path, 'w') as f:
        for r in result:
            f.write(r + '\n')

    print(f"✅ 轉換完成（掃描數={scan_lines}）→ {output_path}")
    # 每行拆成 4 字元一組
    split_lines = [[line[i:i+4] for i in range(0, len(line), 4)] for line in lines]

    result = []

    # 每次處理 scan_lines 行
    for i in range(0, len(split_lines), scan_lines):
        group = split_lines[i:i+scan_lines]
        if len(group) < scan_lines:
            break  # 不足一組就跳出

        num_cols = len(group[0])
        # 對每一欄操作
        for col in range(num_cols):
            # 每兩行一組（下 + 上）
            for row in range(0, scan_lines, 2):
                high = group[row+1][col]
                low = group[row][col]
                result.append(high + low)

    # 輸出
    with open(output_path, 'w') as f:
        for r in result:
            f.write(r + '\n')

    print(f"✅ 轉換完成（掃描數={scan_lines}）→ {output_path}")

In [ ]:
DIR_O = "./ecg_v2_fc2"
_ifm_id         = 4
_stride         = 4
_int16_is       = 1
_pe_out_en      = 1
_ker_w          = 6

stride          = 0
int16_is        = 1
pe_mode         = 1
ker_w           = 4 # 1*5-1
#### PE INFO
instruction_gen(
    total_bits = 32, ## _ifm_id + _ker_w + _stride + _int16_is +_pe_out_en
    fields=[
        ("ifm_id", _ifm_id, "0"),
        ("stride", _stride, str(int(stride))),
        ("int16_is", _int16_is, str(int(int16_is))),
        ("pe_out_en", _pe_out_en, "0"),
        ("pe_mode", _pe_mode, str(int(pe_mode))),
        ("ker_w", _ker_w, str(int(ker_w)))
    ],
    output_file="pe0.txt",
    output_dir=DIR_O
)

✅ 已產生 ./ecg_v2_fc2\pe0.txt


### ECG v2 fc3

## PE TEST

### P_TEST0

In [15]:
CH_I = 8
CH_O = 1
IN_L = 2
KER_W = 1
STRIDE = 1
PAD_L = 0
PAD_R = 0
D_TYPE = 8
BIAS = False
DIR_O = "./pe_test0"

conv_test_gen(
    ch_in=CH_I,
    ch_out=CH_O,
    input_length=IN_L,
    ker_w=KER_W,
    stride=STRIDE,
    pad_left=PAD_L,
    pad_right=PAD_R,
    use_bias=BIAS,
    data_type=D_TYPE,
    ifmap_numbers_per_line=4*8//D_TYPE,
    kernel_numbers_per_line=4*8//D_TYPE,
    bias_numbers_per_line=2,
    golden_numbers_per_line=1,
    random_seed=0,
    input_file = "ifm",
    kernel_file = "ker",
    bias_file = "bias",
    golden_file = "gold",
    output_dir = DIR_O
)
instruction_gen(
    total_bits=4+4+4+1+1, ## 32
    fields=[
        # ("info_id", 5, "0"),
        ("ifm_id", 4, "0"),
        # ("ker_id", 5, "0"),
        # ("p_out_id", 5, "0"),
        ("ker_w", 4, str(int((CH_I*D_TYPE // (8*4))*KER_W-1))), # ker_w need - 1
        ("stride", 4, str(int(STRIDE * (CH_I * (D_TYPE // 8) // 4)))),
        ("int6_is", 1, str(int(D_TYPE // 8 - 1))),
        ("pe_out_en", 1, "0")
    ],
    output_file="pe_config.txt",
    output_dir=DIR_O
)

✅ 檔案產生完成
📁 輸出路徑: c:\Users\N26120579\Desktop\畢業光碟_蔡明翰\3.【系統資料】\Python\Hardware_sim\pe_test0
✅ 已產生 ./pe_test0\pe_config.txt


### PE TEST1

In [16]:
CH_I = 4
CH_O = 1
IN_L = 10
KER_W = 5
STRIDE = 1
PAD_L = 0
PAD_R = 0
D_TYPE = 8
BIAS = False
DIR_O = "./pe_test1"

conv_test_gen(
    ch_in=CH_I,
    ch_out=CH_O,
    input_length=IN_L,
    ker_w=KER_W,
    stride=STRIDE,
    pad_left=PAD_L,
    pad_right=PAD_R,
    use_bias=BIAS,
    data_type=D_TYPE,
    ifmap_numbers_per_line=4*8//D_TYPE,
    kernel_numbers_per_line=4*8//D_TYPE,
    bias_numbers_per_line=2,
    golden_numbers_per_line=1,
    random_seed=0,
    input_file = "ifm",
    kernel_file = "ker",
    bias_file = "bias",
    golden_file = "gold",
    output_dir = DIR_O
)
instruction_gen(
    total_bits=4+4+4+1+1, ## 32
    fields=[
        # ("info_id", 5, "0"),
        ("ifm_id", 4, "0"),
        # ("ker_id", 5, "0"),
        # ("p_out_id", 5, "0"),
        ("ker_w", 4, str(int((CH_I*D_TYPE // (8*4))*KER_W-1))), # ker_w need - 1
        ("stride", 4, str(int(STRIDE * (CH_I * (D_TYPE // 8) // 4)))),
        ("int6_is", 1, str(int(D_TYPE // 8 - 1))),
        ("pe_out_en", 1, "0")
    ],
    output_file="pe_config.txt",
    output_dir=DIR_O
)

✅ 檔案產生完成
📁 輸出路徑: c:\Users\N26120579\Desktop\畢業光碟_蔡明翰\3.【系統資料】\Python\Hardware_sim\pe_test1
✅ 已產生 ./pe_test1\pe_config.txt


### PE TEST 2

In [17]:
CH_I = 4
CH_O = 1
IN_L = 100
KER_W = 3
STRIDE = 1
PAD_L = 0
PAD_R = 0
D_TYPE = 8
BIAS = False
DIR_O = "./pe_test2"

conv_test_gen(
    ch_in=CH_I,
    ch_out=CH_O,
    input_length=IN_L,
    ker_w=KER_W,
    stride=STRIDE,
    pad_left=PAD_L,
    pad_right=PAD_R,
    use_bias=BIAS,
    data_type=D_TYPE,
    ifmap_numbers_per_line=4*8//D_TYPE,
    kernel_numbers_per_line=4*8//D_TYPE,
    bias_numbers_per_line=2,
    golden_numbers_per_line=1,
    random_seed=0,
    input_file = "ifm",
    kernel_file = "ker",
    bias_file = "bias",
    golden_file = "gold",
    output_dir = DIR_O
)
instruction_gen(
    total_bits=4+4+4+1+1,
    fields=[
        # ("info_id", 5, "0"),
        ("ifm_id", 4, "0"),
        # ("ker_id", 5, "0"),
        # ("p_out_id", 5, "0"),
        ("ker_w", 4, str(int((CH_I*D_TYPE // (8*4))*KER_W-1))), # ker_w need - 1
        ("stride", 4, str(int(STRIDE * (CH_I * (D_TYPE // 8) // 4)))),
        ("int6_is", 1, str(int(D_TYPE // 8 - 1))),
        ("pe_out_en", 1, "0")
    ],
    output_file="pe_config.txt",
    output_dir=DIR_O
)

✅ 檔案產生完成
📁 輸出路徑: c:\Users\N26120579\Desktop\畢業光碟_蔡明翰\3.【系統資料】\Python\Hardware_sim\pe_test2
✅ 已產生 ./pe_test2\pe_config.txt


### PE TEST 3

In [18]:
CH_I = 2
CH_O = 1
IN_L = 2
KER_W = 1
STRIDE = 1
PAD_L = 0
PAD_R = 0
D_TYPE = 16
BIAS = False
DIR_O = "./pe_test3"

conv_test_gen(
    ch_in=CH_I,
    ch_out=CH_O,
    input_length=IN_L,
    ker_w=KER_W,
    stride=STRIDE,
    pad_left=PAD_L,
    pad_right=PAD_R,
    use_bias=BIAS,
    data_type=D_TYPE,
    ifmap_numbers_per_line=4*8//D_TYPE,
    kernel_numbers_per_line=4*8//D_TYPE,
    bias_numbers_per_line=2,
    golden_numbers_per_line=1,
    random_seed=0,
    input_file = "ifm",
    kernel_file = "ker",
    bias_file = "bias",
    golden_file = "gold",
    output_dir = DIR_O
)
instruction_gen(
    total_bits=4+4+4+1+1, ## 32
    fields=[
        # ("info_id", 5, "0"),
        ("ifm_id", 4, "0"),
        # ("ker_id", 5, "0"),
        # ("p_out_id", 5, "0"),
        ("ker_w", 4, str(int((CH_I*D_TYPE // (8*4))*KER_W-1))), # ker_w need - 1
        ("stride", 4, str(int(STRIDE * (CH_I * (D_TYPE // 8) // 4)))),
        ("int6_is", 1, str(int(D_TYPE // 8 - 1))),
        ("pe_out_en", 1, "0")
    ],
    output_file="pe_config.txt",
    output_dir=DIR_O
)

✅ 檔案產生完成
📁 輸出路徑: c:\Users\N26120579\Desktop\畢業光碟_蔡明翰\3.【系統資料】\Python\Hardware_sim\pe_test3
✅ 已產生 ./pe_test3\pe_config.txt


### PE TEST 4

In [19]:
CH_I = 2
CH_O = 1
IN_L = 50
KER_W = 2
STRIDE = 1
PAD_L = 0
PAD_R = 0
D_TYPE = 16
BIAS = False
DIR_O = "./pe_test4"

conv_test_gen(
    ch_in=CH_I,
    ch_out=CH_O,
    input_length=IN_L,
    ker_w=KER_W,
    stride=STRIDE,
    pad_left=PAD_L,
    pad_right=PAD_R,
    use_bias=BIAS,
    data_type=D_TYPE,
    ifmap_numbers_per_line=4*8//D_TYPE,
    kernel_numbers_per_line=4*8//D_TYPE,
    bias_numbers_per_line=2,
    golden_numbers_per_line=1,
    random_seed=0,
    input_file = "ifm",
    kernel_file = "ker",
    bias_file = "bias",
    golden_file = "gold",
    output_dir = DIR_O
)
instruction_gen(
    total_bits=4+4+4+1+1, ## 32
    fields=[
        # ("info_id", 5, "0"),
        ("ifm_id", 4, "0"),
        # ("ker_id", 5, "0"),
        # ("p_out_id", 5, "0"),
        ("ker_w", 4, str(int((CH_I*D_TYPE // (8*4))*KER_W-1))), # ker_w need - 1
        ("stride", 4, str(int(STRIDE * (CH_I * (D_TYPE // 8) // 4)))),
        ("int6_is", 1, str(int(D_TYPE // 8 - 1))),
        ("pe_out_en", 1, "0")
    ],
    output_file="pe_config.txt",
    output_dir=DIR_O
)

✅ 檔案產生完成
📁 輸出路徑: c:\Users\N26120579\Desktop\畢業光碟_蔡明翰\3.【系統資料】\Python\Hardware_sim\pe_test4
✅ 已產生 ./pe_test4\pe_config.txt


## golden generator by existed data

In [3]:
import numpy as np

def parse_value(s, base):
    s = s.strip()
    if base == 16:
        return int(s, 16)
    elif base == 8:
        return int(s, 8)
    else:
        return int(s, 10)

def read_ifmap(path, ch_in, ifmap_len, base=10):
    data = []
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                data.extend([parse_value(x, base) for x in line.strip().split()])
    if len(data) != ch_in * ifmap_len:
        raise ValueError(f"ifmap data length {len(data)} != expected {ch_in}x{ifmap_len}")
    return np.array(data, dtype=np.int32).reshape(ch_in, ifmap_len)

def read_kernel(path, ch_out, ch_in, kernel_size, base=10):
    data = []
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                data.extend([parse_value(x, base) for x in line.strip().split()])
    expected = ch_out * kernel_size * ch_in
    if len(data) != expected:
        raise ValueError(f"kernel data length {len(data)} != expected {expected}")
    return np.array(data, dtype=np.int32).reshape(ch_out, kernel_size, ch_in).transpose(0, 2, 1)

def read_bias(path, ch_out, base=10):
    data = []
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                data.append(parse_value(line.strip(), base))
    if len(data) != ch_out:
        raise ValueError(f"bias data length {len(data)} != ch_out {ch_out}")
    return np.array(data, dtype=np.int32)

def save_output(path, data, values_per_line=4):
    with open(path, "w") as f:
        for i in range(0, len(data), values_per_line):
            f.write(" ".join(map(str, data[i:i+values_per_line])) + "\n")

def conv1d_from_txt(
    ifmap_path,
    kernel_path,
    bias_path=None,
    output_path="golden.txt",
    ch_in=4,
    ch_out=2,
    ifmap_len=8,
    kernel_size=3,
    stride=1,
    pad_left=0,
    pad_right=0,
    base_mode_ifmap=10,
    base_mode_kernel=10,
    base_mode_bias=10,
    values_per_line_output=4
):
    # === Load data ===
    ifmap = read_ifmap(ifmap_path, ch_in, ifmap_len, base=base_mode_ifmap)
    kernel = read_kernel(kernel_path, ch_out, ch_in, kernel_size, base=base_mode_kernel)
    if bias_path:
        bias = read_bias(bias_path, ch_out, base=base_mode_bias)
    else:
        bias = np.zeros(ch_out, dtype=np.int32)

    # === Padding ===
    ifmap_padded = np.pad(ifmap, ((0, 0), (pad_left, pad_right)), mode='constant')
    padded_len = ifmap_padded.shape[1]
    out_len = (padded_len - kernel_size) // stride + 1

    # === Convolution ===
    output = np.zeros((ch_out, out_len), dtype=np.int32)
    for oc in range(ch_out):
        for i in range(out_len):
            acc = 0
            for ic in range(ch_in):
                window = ifmap_padded[ic, i*stride : i*stride + kernel_size]
                acc += np.sum(window * kernel[oc, ic])
            acc += bias[oc]
            output[oc, i] = acc

    # === Save ===
    output_flat = output.T.flatten()  # interleave by position
    save_output(output_path, output_flat, values_per_line_output)
    print(f"✅ Saved: {output_path}")

# === Sample test ===
if __name__ == "__main__":
    conv1d_from_txt(
        ifmap_path="ifmap.txt",
        kernel_path="kernel.txt",
        bias_path="bias.txt",
        output_path="golden.txt",
        ch_in=2,
        ch_out=2,
        ifmap_len=4,
        kernel_size=2,
        stride=1,
        pad_left=0,
        pad_right=0,
        base_mode_ifmap=10,
        base_mode_kernel=10,
        base_mode_bias=10,
        values_per_line_output=2
    )


FileNotFoundError: [Errno 2] No such file or directory: 'ifmap.txt'